In [1]:
import pandas as pd
import os
import parmap

In [6]:
def read_csv(args):
    save_dir = "./Dominant_Clone/"
    path = args[0]
    top_x = args[1]
    filename = path.split("/")[-1]
    sample = filename.split("__")[0]
    chain = filename.split("__")[-1].split(".csv")[0]

    save_path = save_dir + sample + "/"
    try:
        if not os.path.exists(save_path):
            os.makedirs(save_path)
    except:
        None

    df = pd.read_csv(path,usecols=["CDR3(pep)","copy"])
    if df.shape[0]<10:
        return (df,sample,chain)
    df = df.groupby("CDR3(pep)").sum().sort_values("copy",ascending=False).iloc[:top_x].reset_index()

    df.to_csv(save_path+chain+".csv",index=False)
    return (df,sample,chain)

In [7]:
top_x = 10
csv_paths = []
for dirname,dirs,filenames in os.walk("./artificial_peps"):
    for filename in filenames:
        csv_paths.append((os.path.join(dirname,filename),top_x))

In [8]:
runtime = parmap.map_async(read_csv,csv_paths, pm_processes=32)
runtime.wait()
result = runtime.get()

In [9]:
Dominant_Clone_Dict = {}
for (df,sample,chain) in result:
    if sample not in Dominant_Clone_Dict.keys():
        Dominant_Clone_Dict[sample] = {chain:df}
    else:
        Dominant_Clone_Dict[sample][chain] = df

In [10]:
chain2Dominant_matrix = {}
for chain in ["TRA","TRB","TRD","TRG","IGH","IGK","IGL"]:
    df_Dominant_matrix = pd.DataFrame(columns=["CDR3(pep)"])
    for sample,chain2df in Dominant_Clone_Dict.items():
        df = chain2df[chain].copy()
        df[sample] = df.pop("copy")
        df_Dominant_matrix = pd.merge(df_Dominant_matrix,df,on="CDR3(pep)",how="outer")
    save_path = "./Dominant_Matrix/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    df_Dominant_matrix.to_csv(save_path+chain+".csv",index=False)
    chain2Dominant_matrix[chain] = df_Dominant_matrix

In [11]:
profile = pd.read_csv("./Profile_All_24_04_22.csv")
use_categorys = ["HHY"]
root_path = "./Dominant_Matrix2Categroy/"
for use_category in use_categorys:
    samples = profile[profile[use_category].isin(profile[use_category].dropna().unique().tolist())]["sample"].tolist()
    save_path = root_path+use_category+"/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    for chain,matrix in chain2Dominant_matrix.items():
        use_matrix = matrix[["CDR3(pep)"]+samples].copy()
        use_matrix.index = use_matrix.pop("CDR3(pep)")
        use_matrix.index.name = "CDR3(pep)"
        use_matrix.fillna(0,inplace=True)
        use_matrix = use_matrix.loc[~(use_matrix==0).all(axis=1)]
        use_matrix = use_matrix.T
        use_matrix.insert(loc=0,column="sample",value=use_matrix.index)
        use_matrix = pd.merge(profile[["sample",use_category]],use_matrix)
        use_matrix.to_csv(save_path+chain+".csv",index=False)

In [110]:
use_matrix

,sample,G1_Blood_Pneumonia_Acute_Recovery,CGTWDSSLSAGVF,CAAWDDSLNGPVF,CSSYTSSSTLVF,CQSYDSSLSGSVF,CAAWDDSLNGWVF,CQVWDSSSDHVVF,CQVWDSSSDHYVF,CQVWDSSSDHWVF,...,CETWDSYTHGVF,CETWDGYTHGVF,CGTWDSSLSAGIF,CETWDGYAHGVF,CGTWDSSLRAGVF,CGTWHSDLSVYVF,CAAWDDSLNGRVF,CCSSAGSYTVVF,CLLFYNGLWVF,CQVWDSSSDSWVF
0,GW_29BCID,Pneumonia-A,182.0,76.0,0.0,75.0,62.0,139.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,GW_30BCID,Pneumonia-R,242.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,650.0,453.0,294.0,228.0,0.0,0.0,0.0,0.0,0.0,0.0
2,GW_31BCID,Pneumonia-A,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,GW_32BCID,Pneumonia-A,141.0,101.0,161.0,102.0,0.0,124.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,148.0,0.0,0.0,0.0,0.0
4,GW_33BCID,Pneumonia-R,0.0,106.0,41.0,0.0,53.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,52.0,69.0,57.0,37.0
5,GW_34BCID,Pneumonia-A,499.0,0.0,0.0,0.0,0.0,389.0,537.0,446.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,GW_41BCID,Pneumonia-A,69.0,0.0,0.0,0.0,56.0,0.0,0.0,78.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,GW_42BCID,Pneumonia-A,275.0,0.0,171.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,GW_50BCID,Pneumonia-A,0.0,148.0,192.0,203.0,146.0,175.0,0.0,163.0,...,0.0,0.0,0.0,0.0,160.0,0.0,0.0,0.0,0.0,0.0
